In [1]:
import socket
print(socket.gethostname())

awr-2-15


In [2]:
import xarray


### Pressure & Temperature

* **Mean sea level pressure (slp):** `mean_sea_level_pressure`
* **Surface pressure (p_sfc):** `surface_pressure`
* **2-m dewpoint temperature (Td_2m):** `2m_dewpoint_temperature`
* **2-m temperature (T_2m):** `2m_temperature`
* **Skin temperature (T_sfc):** `skin_temperature`
* **Sea surface temperature:** `sea_surface_temperature` *(Note: In ERA5, Skin Temperature is often used over land/ice, while SST is used over open ocean).*

### Moisture & Transport

* **Integrated water vapor (IWV):** `total_column_water_vapour`
* **Integrated vapor transport (meridional) (IVTV):** `vertical_integral_of_northward_water_vapour_flux`
* **Integrated vapor transport (zonal) (IVTU):** `vertical_integral_of_eastward_water_vapour_flux`

### Wind & Precipitation

* **10-m zonal wind (u_10m_gr):** `10m_u_component_of_wind`
* **10-m meridional wind (v_10m_gr):** `10m_v_component_of_wind`
* **6-hourly accumulated precipitation (precip_bkt):** `total_precipitation`

### Forcing
* **Geopotential at the surface (Z_sfc):** `geopotential_at_surface`

---

### Summary Table for Quick Reference

| Your Variable | ERA5 Equivalent Name |
| --- | --- |
| **slp** | `mean_sea_level_pressure` |
| **p_sfc** | `surface_pressure` |
| **Td_2m** | `2m_dewpoint_temperature` |
| **T_2m** | `2m_temperature` |
| **T_sfc** | `skin_temperature` / `sea_surface_temperature` |
| **IWV** | `total_column_water_vapour` |
| **IVTU / IVTV** | `vertical_integral_of_eastward/northward_water_vapour_flux` |
| **u_10m / v_10m** | `10m_u_component_of_wind` / `10m_v_component_of_wind` |
| **precip_bkt** | `total_precipitation` |
| **Z_sfc** |  `geopotential_at_surface` |

### Pressure-Level Variables (3D)

| Your Variable | ERA5 Equivalent Name | Dimensions |
| --- | --- | --- |
| **Geopotential (Z_e)** | `geopotential` | (time, level, latitude, longitude) |
| **Temperature (T_e)** | `temperature` | (time, level, latitude, longitude) |
| **Specific humidity (q_e)** | `specific_humidity` | (time, level, latitude, longitude) |
| **Zonal wind (u_gr_e)** | `u_component_of_wind` | (time, level, latitude, longitude) |
| **Meridional wind (v_gr_e)** | `v_component_of_wind` | (time, level, latitude, longitude) |



In [19]:
ds = xarray.open_zarr(
    'gs://gcp-public-data-arco-era5/ar/model-level-1h-0p25deg.zarr-v1',
    chunks=None,
    storage_options=dict(token='anon'),
)
ar_native_vertical_grid_data = ds.sel(time=slice('2019-01-01T00:00:00.000000000', '2019-01-01T23:00:00.000000000'))

In [20]:
ds = xarray.open_zarr(
    'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3',
    chunks=None,
    storage_options=dict(token='anon'),
)
ar_full_37_1h = ds.sel(time=slice('2019-01-01T00:00:00.000000000', '2019-01-01T23:00:00.000000000'))

ar_model_level_and_surface_data = xarray.merge([
    ar_native_vertical_grid_data, ar_full_37_1h.drop_dims('level')
])

In [ ]:
import xarray as xr

# 1. Define the variables we want to keep (using ERA5 names)
selected_vars = [
    "mean_sea_level_pressure", "surface_pressure", "2m_dewpoint_temperature",
    "2m_temperature", "skin_temperature", "total_column_water_vapour",
    "vertical_integral_of_eastward_water_vapour_flux", 
    "vertical_integral_of_northward_water_vapour_flux",
    "10m_u_component_of_wind", "10m_v_component_of_wind", 
    "total_precipitation", "geopotential", "temperature", 
    "specific_humidity", "u_component_of_wind", "v_component_of_wind",
    "geopotential_at_surface"
]

# 2. Select and Rename to match your desired shorthand
rename_map = {
    "mean_sea_level_pressure": "slp",
    "surface_pressure": "p_sfc",
    "2m_dewpoint_temperature": "Td_2m",
    "2m_temperature": "T_2m",
    "skin_temperature": "T_sfc", #TODO: confirm if this is the correct variable for skin temperature
    "total_column_water_vapour": "IWV",
    "vertical_integral_of_eastward_water_vapour_flux": "IVTU",
    "vertical_integral_of_northward_water_vapour_flux": "IVTV",
    "10m_u_component_of_wind": "u_10m_gr",
    "10m_v_component_of_wind": "v_10m_gr",
    "total_precipitation": "precip_bkt",
    "geopotential": "Z_e",
    "temperature": "T_e",
    "specific_humidity": "q_e",
    "u_component_of_wind": "u_gr_e",
    "v_component_of_wind": "v_gr_e",
    "geopotential_at_surface": "Z_sfc"
}

# Apply selection and renaming
ds_combined = ar_model_level_and_surface_data[selected_vars].rename(rename_map)

# 3. Resample to 6-hour intervals
# For precipitation, use sum
precip_resampled = ds_combined['precip_bkt'].resample(time='6h').sum()

# For other variables, use mean
others = ds_combined.drop_vars('precip_bkt')
others_resampled = others.resample(time='6h').mean()

# 4. Merge back the resampled data
ds_resampled = xr.merge([others_resampled, precip_resampled])

# 5. Save to NetCDF
save_path = '/cw3e/mead/projects/cwp167/moerfani_data/global/2019/01/era5_modellev_d01_2019-01-01.nc'
ds_resampled.to_netcdf(save_path)

print(f"File successfully saved to {save_path}")

File successfully saved to /cw3e/mead/projects/cwp167/moerfani_data/global/2019/01/era5_modellev_d01_2019-01-01.nc
